# api-ops-full — 全量专轨

**环境**：Jupyter 内核选 **`med-rag-verify`**（与 01–11 相同）。  
F0 建库只用标准库 `sqlite3` + 本仓库代码，**不需要** Ollama / Chroma / GPU。  
若缺包：在阶段 12 目录执行 `pip install -r requirements.txt`（主要补 `python-dotenv`）。

- **F0**：构建 / 续跑 `Dataset/documents/full/documents_full.sqlite`（进度可视化）
- **F1+**：全量仿真（阶段 5；需 `full/manifest_full.json` → `status=completed`）

与样本分目录：`documents/sample/` ≠ `documents/full/`。

也可终端后台（先 `conda activate med-rag-verify`）：
```text
python scripts/build_documents_index.py --mode full --batch-size 50000
python scripts/build_documents_index.py --mode full --status
```


## F0 — 环境隔离 + 全量目录自检

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
STAGE12 = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REPO = STAGE12.parent

for name in ("config", "bootstrap", "resources", "app"):
    sys.modules.pop(name, None)
for key in list(sys.modules):
    if key == "app" or key.startswith("app."):
        sys.modules.pop(key, None)

for p in (str(REPO), str(STAGE12)):
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(STAGE12))

from app.bootstrap import bootstrap_paths
from app.bridge11 import reset_stage11_cache
from app.documents_index import (
    build_documents_index,
    manifest_path_for,
    mode_dir_for,
    progress_path_for,
    sqlite_path_for,
    status,
)
from dataset_paths import DOCUMENTS_FULL_DIR, DOCUMENTS_FULL_SQLITE, SLIM_JSONL

reset_stage11_cache()
bootstrap_paths(STAGE12)

print("slim", SLIM_JSONL.exists(), SLIM_JSONL)
print("full_dir", DOCUMENTS_FULL_DIR)
print("sqlite", DOCUMENTS_FULL_SQLITE)
print("progress", progress_path_for("full"))
print("manifest", manifest_path_for("full"))
print(json.dumps(status("full"), ensure_ascii=False, indent=2))

slim True D:\谷歌\Dataset\processed\oa_comm_slim.jsonl
full_dir D:\谷歌\Dataset\documents\full
sqlite D:\谷歌\Dataset\documents\full\documents_full.sqlite
progress D:\谷歌\Dataset\documents\full\progress_full.json
manifest D:\谷歌\Dataset\documents\full\manifest_full.json
{
  "mode": "full",
  "sqlite": "D:\\谷歌\\Dataset\\documents\\full\\documents_full.sqlite",
  "sqlite_exists": false,
  "row_count": null,
  "progress": {},
  "manifest": {},
  "completed": false
}


## F0 — 启动 / 续跑全量构建

将 `RUN_FULL_BUILD = True` 后运行。默认 `RESUME=True`（中断可续）。
产物全部写入 `Dataset/documents/full/`。

In [2]:
# >>> 创建全量索引：改为 True 后运行本格 <<<
RUN_FULL_BUILD = True
BATCH_SIZE = 50_000
RESUME = True  # 中断后续跑同一格即可；强制从头则 False

mode_dir_for("full").mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
history = []

def _cb(p):
    history.append(dict(p))
    lines = p.get("processed_lines") or 0
    rows = p.get("valid_rows") or 0
    print(
        f"[{p.get('phase')}] lines={lines:,} rows={rows:,} "
        f"last={p.get('last_pmcid')} elapsed={time.perf_counter()-t0:.1f}s",
        flush=True,
    )

if RUN_FULL_BUILD:
    print("building →", sqlite_path_for("full"))
    manifest = build_documents_index(
        "full",
        batch_size=BATCH_SIZE,
        resume=RESUME,
        progress_cb=_cb,
    )
    print(json.dumps(manifest, ensure_ascii=False, indent=2))
    assert manifest.get("status") == "completed"
    print("F0 PASS — documents/full ready")
else:
    print("RUN_FULL_BUILD=False — 仅展示 status")
    print(json.dumps(status("full"), ensure_ascii=False, indent=2))

building → D:\谷歌\Dataset\documents\full\documents_full.sqlite
[writing] lines=50,000 rows=50,000 last=PMC2377171 elapsed=1.1s
[writing] lines=100,000 rows=100,000 last=PMC2830950 elapsed=2.3s
[writing] lines=150,000 rows=150,000 last=PMC3044096 elapsed=3.4s
[writing] lines=200,000 rows=200,000 last=PMC3250972 elapsed=4.5s
[writing] lines=250,000 rows=250,000 last=PMC3435393 elapsed=5.7s
[writing] lines=300,000 rows=300,000 last=PMC3591258 elapsed=6.9s
[writing] lines=350,000 rows=350,000 last=PMC3751602 elapsed=8.2s
[writing] lines=400,000 rows=400,000 last=PMC3907692 elapsed=9.5s
[writing] lines=450,000 rows=450,000 last=PMC4053804 elapsed=10.7s
[writing] lines=500,000 rows=500,000 last=PMC4214501 elapsed=12.0s
[writing] lines=550,000 rows=550,000 last=PMC4363399 elapsed=13.4s
[writing] lines=600,000 rows=600,000 last=PMC4496389 elapsed=14.7s
[writing] lines=650,000 rows=650,000 last=PMC4632024 elapsed=16.1s
[writing] lines=700,000 rows=700,000 last=PMC4762005 elapsed=17.4s
[writing] 

In [3]:
# 可选：另一终端 CLI 构建时，在此轮询 progress_full.json
POLL = False
POLL_ROUNDS = 60
POLL_SEC = 10

if POLL:
    try:
        from IPython.display import clear_output
    except ImportError:
        clear_output = None
    for i in range(POLL_ROUNDS):
        st = status("full")
        if clear_output:
            clear_output(wait=True)
        print(f"poll {i+1}/{POLL_ROUNDS}")
        print(json.dumps(st, ensure_ascii=False, indent=2))
        if st.get("completed"):
            print("completed")
            break
        time.sleep(POLL_SEC)
else:
    print("POLL=False — 需要监视 CLI 进度时改为 True")
    print(json.dumps(status("full"), ensure_ascii=False, indent=2))

POLL=False — 需要监视 CLI 进度时改为 True
{
  "mode": "full",
  "sqlite": "D:\\谷歌\\Dataset\\documents\\full\\documents_full.sqlite",
  "sqlite_exists": true,
  "row_count": 4557627,
  "progress": {
    "format": "documents_index_v1",
    "mode": "full",
    "schema_version": 1,
    "batch_size": 50000,
    "processed_lines": 4557627,
    "valid_rows": 4557627,
    "last_pmcid": "PMC12823268",
    "matched_sample": null,
    "sample_target": null,
    "updated_at": "2026-07-27T08:16:08Z",
    "source_path": "D:\\谷歌\\Dataset\\processed\\oa_comm_slim.jsonl",
    "source_exists": true,
    "source_size_bytes": 8896642264,
    "source_mtime": 1779813364.2908616,
    "phase": "completed",
    "status": "completed",
    "row_count": 4557627
  },
  "manifest": {
    "format": "documents_index_v1",
    "status": "completed",
    "mode": "full",
    "schema_version": 1,
    "batch_size": 50000,
    "row_count": 4557627,
    "valid_rows_written": 4557627,
    "processed_lines": 4557627,
    "elapsed_sec":

## F1+ — 全量仿真（阶段 5）

待 `Dataset/documents/full/manifest_full.json` 的 `status=completed`，且阶段 4 完成后补齐。